# KKBOX 표본 고객 100,000명 선정

## 목적

- 프로젝트에서 사용할 고객 100,000명을 선정한다.
- 팀원 모두 동일한 고객을 기준으로 전처리, EDA, 피처 엔지니어링을 진행한다.
- msno를 공통 고객 식별자로 사용한다.
- 원본 데이터의 is_churn 비율을 유지하여 표본을 추출한다.

## 표본 추출 기준

- 대상 고객: 아래 4개 데이터에 모두 존재하는 고객
    - train_v2.csv
    - members_v3.csv
    - transactions_v2.csv
    - user_logs_v2.csv
- 고객 식별 컬럼: msno
- 타깃 컬럼: is_churn
- 표본 크기: 100,000명
- 추출 방법: is_churn 기준 층화 추출
- 난수 고정값: random_state=42

## 프로젝트 예측 기준

- 관찰 종료일: 2017-02-28
- 이탈 예측 기간: 2017-03-01 ~ 2017-03-31

# 1. 세 파일 공통 고객 찾기

In [2]:
import pandas as pd
# 데이터 불러오기
members = pd.read_csv("../data/raw/members_v3.csv")
transactions = pd.read_csv("../data/raw/transactions_v2.csv")
user_logs = pd.read_csv("../data/raw/user_logs_v2.csv")
target = pd.read_csv("../data/raw/train_v2.csv")

# 각 데이터에서 고객 ID 추출
members_users = set(members["msno"].dropna().unique())
transactions_users = set(transactions["msno"].dropna().unique())
user_logs_users = set(user_logs["msno"].dropna().unique())
target_users = set(target["msno"].dropna().unique())

# 네 데이터에 모두 존재하는 고객
common_users = (
    members_users
    & transactions_users
    & user_logs_users
    & target_users
)

print(f"공통 고객 수: {len(common_users):,}명")

공통 고객 수: 725,722명


# 2. 10만명 추출

In [3]:
common_target = (
    target[target["msno"].isin(common_users)]
    [["msno", "is_churn"]]
    .dropna(subset=["is_churn"])
    .drop_duplicates(subset="msno")
)

print(common_target.shape)
print(common_target["is_churn"].value_counts())
print(common_target["is_churn"].value_counts(normalize=True))

(725722, 2)
is_churn
0    679119
1     46603
Name: count, dtype: int64
is_churn
0    0.935784
1    0.064216
Name: proportion, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split

if len(common_target) < 100_000:
    raise ValueError(
        f"공통 고객이 {len(common_target):,}명이라 10만 명을 추출할 수 없습니다."
    )

target_100k, _ = train_test_split(
    common_target,
    train_size=100_000,
    stratify=common_target["is_churn"],
    random_state=42
)

selected_msno = set(target_100k["msno"])

print(f"선정 고객 수: {len(selected_msno):,}명")
print(target_100k["is_churn"].value_counts())
print(target_100k["is_churn"].value_counts(normalize=True))

선정 고객 수: 100,000명
is_churn
0    93578
1     6422
Name: count, dtype: int64
is_churn
0    0.93578
1    0.06422
Name: proportion, dtype: float64


# 3. 선정된 고객만 추출

In [5]:
members_100k = members[
    members["msno"].isin(selected_msno)
].copy()

transactions_100k = transactions[
    transactions["msno"].isin(selected_msno)
].copy()

user_logs_100k = user_logs[
    user_logs["msno"].isin(selected_msno)
].copy()

In [6]:
print("target:", target_100k.shape)
print("members:", members_100k.shape)
print("transactions:", transactions_100k.shape)
print("user_logs:", user_logs_100k.shape)

print("members 고객 수:", members_100k["msno"].nunique())
print("transactions 고객 수:", transactions_100k["msno"].nunique())
print("user_logs 고객 수:", user_logs_100k["msno"].nunique())

target: (100000, 2)
members: (100000, 6)
transactions: (122479, 9)
user_logs: (1815285, 9)
members 고객 수: 100000
transactions 고객 수: 100000
user_logs 고객 수: 100000


# 4. 파일 저장

In [8]:
from pathlib import Path

# 노트북이 프로젝트 최상위 폴더에 있는 경우
save_path = Path("../data/sampled")
save_path.mkdir(parents=True, exist_ok=True)

target_100k.to_csv(
    save_path / "train_100k.csv",
    index=False
)

members_100k.to_csv(
    save_path / "members_100k.csv",
    index=False
)

transactions_100k.to_csv(
    save_path / "transactions_100k.csv",
    index=False
)

user_logs_100k.to_csv(
    save_path / "user_logs_100k.csv",
    index=False
)

print(f"저장 완료: {save_path.resolve()}")

저장 완료: C:\dev\project\2차 단위 프로젝트\2nd-feature-data\data\sampled
